# Example 9.1 / 2: Map Keys and Data Columns

Companion notebook for **9.1 The `table` Statement** in the AMPL lab Documentation, section "Map Keys and Data Columns". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Map Keys and Data Columns

For data indexed by two dimensions, list two key columns. `market.csv` holds one row per
product-period pair:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
set PERIOD;                       # planning periods
param market {PROD, PERIOD} >= 0; # maximum sales by product and period

table MarketData IN "amplcsv" "market.csv":
    [product, period], market;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
with open("market.csv", "w") as f:
    f.write("product,period,market\n"
            "bands,1,6000\nbands,2,5000\n"
            "coils,1,4000\ncoils,2,4500\n"
            "plate,1,3500\nplate,2,3000\n")
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.set["PERIOD"] = [1, 2]

In [5]:
# run.py
ampl.eval("load amplcsv.dll;")
ampl.read_table("MarketData")
print(ampl.param["market"].to_dict())

{('bands', 1): 6000, ('bands', 2): 5000, ('coils', 1): 4000, ('coils', 2): 4500, ('plate', 1): 3500, ('plate', 2): 3000}


### Pure AMPL: data.dat and model.run

Pure AMPL, with the same `model.mod` and `market.csv`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
set PERIOD := 1 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
load amplcsv.dll;
read table MarketData;
display market;

market :=
bands 1   6000
bands 2   5000
coils 1   4000
coils 2   4500
plate 1   3500
plate 2   3000
;

